# 28.3 — TripletAE + Recon + Corpus Hard-Negative Mining (WJ 512)

Extension of **nb28** with three recall-focused changes:

| Change | nb28 | nb28_3 |
|--------|------|--------|
| Architecture | `TripletEncoder` (no decoder) | **`TripletAE`** + recon (`λ=0.1`) |
| Positives | `max_pos=30` | **`max_pos=100`** |
| Hard negatives | In-batch only (~1.1% corpus/step) | **WJ HNSW corpus mining** → explicit triplets |

**Not repeated here:** nb28_2 hnswlib fast eval (already done), efSearch sweeps (flat R@50).

**Success targets (full, Stage-1):** beat nb28 R@50 **0.656**; ideally beat rand proj **0.663**.

**Run order:** Cell 2 (status) → Phase A nohup warmup → Phase B mine → Phase C nohup finetune → Phase D eval → Phase E compare baselines.

## Prerequisites

Before starting, these should exist:
- `/tmp/best_sota_triplet_autoencoder_wj_512_full.pt` — nb28 encoder (warmup init)
- `/tmp/qt_norm_full.npy` — cached L1-simplex vectors (~16 GB, auto-built on first run)
- `/tmp/gt_lookup_full.pkl` — ground truth
- 8 GPUs recommended; vectors load on **cuda:7**, model on **cuda:0** (DataParallel)

Artifacts this notebook creates (won't overwrite nb28):
- Ckpt: `/tmp/best_triplet_ae_mined_wj_512_full.pt`
- Triplets: `/tmp/triplets_mined_full.pkl`
- Results: `/tmp/results_triplet_ae_mined_wj_512.pkl`

## Cell 1 — Config + artifact status

Single source of truth for hyperparameters. Run this first every session.

In [ ]:
import subprocess, sys
from pathlib import Path

PROJ = Path("/raid/ruban/hpmlproj/term_project")
RUNNER = PROJ / "run_28_3_phase.py"

# ── hyperparameters (also defaults in run_28_3_phase.py) ──
WARMUP_EPOCHS  = 20
FINETUNE_EPOCHS = 15
MAX_POS        = 100
LAMBDA_RECON   = 0.1
HARD_POOL_K    = 500

NB28_CKPT  = "/tmp/best_sota_triplet_autoencoder_wj_512_full.pt"
CKPT_28_3  = "/tmp/best_triplet_ae_mined_wj_512_full.pt"
TRIPLETS   = "/tmp/triplets_mined_full.pkl"
OUT_PKL    = "/tmp/results_triplet_ae_mined_wj_512.pkl"
LOG_WARMUP = "/tmp/nb28_3_warmup.log"
LOG_FT     = "/tmp/nb28_3_finetune.log"

print("nb28 ckpt:", "OK" if Path(NB28_CKPT).exists() else "MISSING — run nb28 first")
subprocess.run([sys.executable, str(RUNNER), "--phase", "status"], check=False)

## Phase A — Warmup (TripletAE + recon, max_pos=100)

- Initializes **encoder** weights from nb28 ckpt; decoder trains from scratch.
- In-batch WJ triplet + FN mask + anchor reconstruction.
- ~4M pairs → ~2,000 steps/epoch × 20 epochs ≈ **4–6 hours**.

Launch with **nohup** (recommended). Monitor with `tail -f`.

In [ ]:
# Launch warmup in background (nohup). Skip if already running.
import os
cmd = (
    f"nohup {sys.executable} {RUNNER} --phase warmup "
    f"--warmup_epochs {WARMUP_EPOCHS} "
    f"> {LOG_WARMUP} 2>&1 &"
)
print("Command:", cmd)
print("\nTo run: uncomment the next line, or paste the command in a terminal.")
# os.system(cmd)
print(f"\nMonitor: tail -f {LOG_WARMUP}")

### Phase A — Resume warmup (optional)

If warmup was interrupted, re-launch with `--resume` to continue from the partial 28_3 ckpt.

In [ ]:
cmd_resume = (
    f"nohup {sys.executable} {RUNNER} --phase warmup "
    f"--warmup_epochs {WARMUP_EPOCHS} --resume "
    f">> {LOG_WARMUP} 2>&1 &"
)
print(cmd_resume)
# os.system(cmd_resume)

### Phase A — Check warmup progress

In [ ]:
import subprocess
if Path(LOG_WARMUP).exists():
    subprocess.run(["tail", "-20", LOG_WARMUP])
else:
    print(f"No log yet at {LOG_WARMUP}")
subprocess.run([sys.executable, str(RUNNER), "--phase", "status"])

## Phase B — Mine corpus hard negatives

- Embeds full dataset with current 28_3 model.
- WJ HNSW top-500 candidates per query → ~440K explicit (q, pos, neg) triplets.
- Takes ~20–30 minutes. **Run in notebook** (fine for this duration).

Requires warmup ckpt to exist.

In [ ]:
assert Path(CKPT_28_3).exists(), f"Run Phase A first — missing {CKPT_28_3}"
subprocess.run(
    [sys.executable, str(RUNNER), "--phase", "mine"],
    check=True,
)

## Phase C — Finetune on mined triplets

- Explicit WJ triplet loss + anchor recon.
- Re-mines hard negatives every 5 epochs.
- 15 epochs ≈ **2–3 hours**. Use nohup.

In [ ]:
assert Path(TRIPLETS).exists(), f"Run Phase B first — missing {TRIPLETS}"
cmd_ft = (
    f"nohup {sys.executable} {RUNNER} --phase finetune "
    f"--finetune_epochs {FINETUNE_EPOCHS} "
    f"> {LOG_FT} 2>&1 &"
)
print(cmd_ft)
# os.system(cmd_ft)
print(f"\nMonitor: tail -f {LOG_FT}")

### Phase C — Check finetune progress

In [ ]:
if Path(LOG_FT).exists():
    subprocess.run(["tail", "-20", LOG_FT])
else:
    print(f"No log yet at {LOG_FT}")

## Phase D — Eval (nmslib WJ HNSW + rerank)

Paper protocol: nmslib WeightedJaccard HNSW, efSearch=200, rerank@1k and @2k.
Takes ~15–20 minutes. Run after finetune completes.

In [ ]:
assert Path(CKPT_28_3).exists(), f"Missing ckpt {CKPT_28_3}"
subprocess.run(
    [sys.executable, str(RUNNER), "--phase", "eval"],
    check=True,
)

## Phase E — Compare vs baselines (no re-run)

Reads existing result pickles. Does not re-execute nb28 or nb28_2.

In [ ]:
import pickle

BASELINES = {
    "nb28 (triplet only)": "/tmp/results_sota_triplet_autoencoder_wj_512.pkl",
    "rand proj":           "/tmp/results_sota_random_proj_wj_512.pkl",
    "nb28_2 hnswlib":      "/tmp/results_sota_triplet_wj_512_fast.pkl",
    "nb28_3 (this run)":   OUT_PKL,
}

print(f"{'method':<28} {'split':<6} {'R@50':>8} {'QPS':>10}  rerank?")
print("-" * 62)
for label, path in BASELINES.items():
    if not Path(path).exists():
        print(f"{label:<28}  (missing {path})")
        continue
    with open(path, "rb") as f:
        d = pickle.load(f)
    if "full" not in d:
        continue
    for k, v in sorted(d["full"].items()):
        if not isinstance(v, dict) or 50 not in v:
            continue
        rr = "rerank" in k
        print(f"{label:<28} {'full':<6} {v[50]:>8.4f} {v.get('qps', 0):>10.0f}  {rr}")
        break  # first HNSW-only row per pickle; rerank rows printed below
    for k, v in sorted(d["full"].items()):
        if not isinstance(v, dict) or 50 not in v or "rerank" not in k:
            continue
        print(f"{label:<28} {'full':<6} {v[50]:>8.4f} {v.get('qps', 0):>10.0f}  rerank@{v.get('candidate_k', '?')}")

## Optional — QPS tradeoff (after recall is good)

Once 28_3 eval is done, compare inference stacks using **nb28_2** on the new ckpt:
1. Copy new ckpt → run nb28_2 eval cell with updated `NB28_CKPT` path
2. Sweep `candidate_k` (1000 vs 2000) for rerank QPS/recall tradeoff

efSearch is **not** useful here (flat R@50 on nb28 embeddings).